## SPARQL Playground

### 1. Retrieve & reconstruction item

In [1]:
from queries.sparql_queries import RETRIEVE_OBJECT_QUERY

In [2]:
from rdflib import Graph
from rdflib.namespace import RDF, RDFS, XSD
from collections import defaultdict

from typing import Optional

from metamodel import Item, LinkedEntity

def fetch_and_wrap_item(identifier: str, rdf_path: str) -> Optional[Item]:
    """
    Fetches data for a given identifier from RDF data via SPARQL and wraps it
    into an Item Pydantic object.

    Args:
        identifier: The unique identifier of the item to fetch.
        rdf_data: A string containing the RDF triples in Turtle format.

    Returns:
        An Item Pydantic object if the item is found, otherwise None.
    """
    g = Graph()
    g.parse(rdf_path, format="turtle")

    sparql_query = RETRIEVE_OBJECT_QUERY % identifier
    
    results = g.query(sparql_query)

    # Aggregate results for multi-valued fields
    item_data = defaultdict(lambda: {'uris': set(), 'labels': set()})
    
    # Store single-value fields directly for the first row, assuming they are consistent
    single_value_fields = {}
    lending_info = {}

    found_item = False

    for row in results:
        found_item = True
        # Direct fields
        if row.identifier:
            single_value_fields['identifier'] = str(row.identifier)
        if row.name:
            single_value_fields['name'] = str(row.name)
        if row.description:
            single_value_fields['description'] = str(row.description)
        if row.caption:
            single_value_fields['caption'] = str(row.caption)
        if row.caseNumber:
            single_value_fields['case_number'] = str(row.caseNumber)
        if row.image:
            single_value_fields['image'] = str(row.image)
        if row.audio:
            single_value_fields['audio'] = str(row.audio)
        if row.video:
            single_value_fields['video'] = str(row.video)
        if row.url:
            single_value_fields['url'] = str(row.url)


        # Category (single LinkedEntity, but collected as if list then take first)
        if row.categoryUri and row.categoryLabel:
            item_data['category']['uris'].add(str(row.categoryUri))
            item_data['category']['labels'].add(str(row.categoryLabel))
        
        # Type (single LinkedEntity, but collected as if list then take first)
        if row.typeUri and row.typeLabel:
            item_data['type']['uris'].add(str(row.typeUri))
            item_data['type']['labels'].add(str(row.typeLabel))


        # List of LinkedEntities
        if row.exhibitionUri and row.exhibitionLabel:
            item_data['exhibition']['uris'].add(str(row.exhibitionUri))
            item_data['exhibition']['labels'].add(str(row.exhibitionLabel))
        
        if row.genreUri and row.genreLabel:
            item_data['genre']['uris'].add(str(row.genreUri))
            item_data['genre']['labels'].add(str(row.genreLabel))
        
        if row.musicWorkUri and row.musicWorkLabel:
            item_data['music_work']['uris'].add(str(row.musicWorkUri))
            item_data['music_work']['labels'].add(str(row.musicWorkLabel))
            
        if row.musicAlbumUri and row.musicAlbumLabel:
            item_data['music_album']['uris'].add(str(row.musicAlbumUri))
            item_data['music_album']['labels'].add(str(row.musicAlbumLabel))
            
        if row.musicArtistUri and row.musicArtistLabel:
            item_data['music_artist']['uris'].add(str(row.musicArtistUri))
            item_data['music_artist']['labels'].add(str(row.musicArtistLabel))

        if row.eventUri and row.eventLabel:
            item_data['event']['uris'].add(str(row.eventUri))
            item_data['event']['labels'].add(str(row.eventLabel))

        if row.countryUri and row.countryLabel:
            item_data['country']['uris'].add(str(row.countryUri))
            item_data['country']['labels'].add(str(row.countryLabel))
        
        if row.placeUri and row.placeLabel:
            item_data['place']['uris'].add(str(row.placeUri))
            item_data['place']['labels'].add(str(row.placeLabel))
        
        if row.venueUri and row.venueLabel:
            item_data['venue']['uris'].add(str(row.venueUri))
            item_data['venue']['labels'].add(str(row.venueLabel))

        if row.colourUri and row.colourLabel:
            item_data['colour']['uris'].add(str(row.colourUri))
            item_data['colour']['labels'].add(str(row.colourLabel))
        
        if row.materialUri and row.materialLabel:
            item_data['material']['uris'].add(str(row.materialUri))
            item_data['material']['labels'].add(str(row.materialLabel))

        # Lending information
        if row.lender:
            lending_info['lender'] = str(row.lender)
        if row.lendStart:
            lending_info['lend_start'] = str(row.lendStart)
        if row.lendEnd:
            lending_info['lend_end'] = str(row.lendEnd)

    if not found_item:
        return None

    # Final structure for Pydantic model
    pydantic_item_fields = {**single_value_fields, **lending_info}

    # Convert sets to lists of LinkedEntity objects
    for field, values in item_data.items():
        if field in ['category', 'type']: # Single LinkedEntity fields
            if values['uris'] and values['labels']:
                pydantic_item_fields[field] = LinkedEntity(
                    uri=list(values['uris'])[0], 
                    label=list(values['labels'])[0]
                )
            elif values['labels']: # If only label is present
                 pydantic_item_fields[field] = LinkedEntity(label=list(values['labels'])[0])
            else:
                pydantic_item_fields[field] = None # Ensure it's None if no data
        else: # List of LinkedEntity fields
            linked_entities_list = []
            # For list fields, create a list of LinkedEntity objects.
            # We need to correctly pair URI and Label. A simple way is to assume
            # a 1-to-1 correspondence based on results or re-query for each URI.
            # For simplicity, we'll create a dictionary to map URI to Label
            # and then reconstruct, ensuring unique (URI, Label) pairs.
            unique_entities = set()
            for uri, label in zip(values['uris'], values['labels']):
                unique_entities.add((uri, label))

            if not unique_entities and values['labels']: # Handle cases where only labels are present for list
                 for label in values['labels']:
                     linked_entities_list.append(LinkedEntity(label=label))
            else:
                for uri, label in unique_entities:
                    linked_entities_list.append(LinkedEntity(uri=uri, label=label))
            
            if linked_entities_list:
                pydantic_item_fields[field] = linked_entities_list
            else:
                pydantic_item_fields[field] = None


    # Handle the special case for 'category' which is a mandatory field in Item.
    # If not found in SPARQL results, it's an error in data or query, but Pydantic expects it.
    if 'category' not in pydantic_item_fields or pydantic_item_fields['category'] is None:
        raise ValueError("Category is a mandatory field and was not found for the item.")
    
    # Handle the special case for 'name' which is a mandatory field in Item.
    if 'name' not in pydantic_item_fields or pydantic_item_fields['name'] is None:
        raise ValueError("Name is a mandatory field and was not found for the item.")

    try:
        item = Item(**pydantic_item_fields)
        return item
    except Exception as e:
        print(f"Error creating Pydantic Item: {e}")
        print(f"Data attempting to pass: {pydantic_item_fields}")
        return None

In [3]:
example_rdf_data = "examples/testset_proto_3.ttl"

In [5]:
# List of identifiers for the 7 objects
object_identifiers = [
    "a4d8c1fe", # Bryan May's Red Special
    "b9e1f3a0", # Sex Pistols 'Anarchy' Muslin Shirt
    "c5f0b2d3", # Master Tape for 'Sgt. Pepper's Lonely Hearts Club Band'
    "d1e4c8a9", # Handwritten Lyrics for 'Wonderwall'
    "e6f7b9c2", # Spice Girls 'Spice World' Lunchbox
    "f8a0d7e4", # John Lennon's Round 'Granny' Glasses
    "g2c5h4b7"  # BRIT Award for British Album of the Year (2012)
]

print("--- Fetching and Wrapping Items ---")
for obj_id in object_identifiers:
    print(f"\nProcessing object with identifier: {obj_id}")
    item_pydantic = fetch_and_wrap_item(obj_id, example_rdf_data)
    
    if item_pydantic:
        print(item_pydantic.model_dump_json(indent=2))
        is_final, errors = item_pydantic.is_final()
        print(f"Is item final? {is_final}")
        if not is_final:
            print(f"Missing URIs for: {', '.join(errors)}")
    else:
        print(f"No item found or error processing for identifier: {obj_id}")
    break  # remove to run it all
print("--- End of Processing ---")

--- Fetching and Wrapping Items ---

Processing object with identifier: a4d8c1fe
{
  "identifier": "a4d8c1fe",
  "name": "Bryan May's Red Special",
  "category": {
    "uri": "http://www.wikidata.org/entity/Q34379",
    "label": "Musical Instrument"
  },
  "type": {
    "uri": "http://www.mimo-db.eu/InstrumentsKeywords/3236",
    "label": "Electric Guitar"
  },
  "description": "The iconic homemade electric guitar built by Queen's guitarist Brian May and his father.",
  "caption": "Famously crafted from an 18th-century fireplace mantel, it was instrumental in creating Queen's unique sound. The guitar that powered the sound of Queen, displayed in the Hard Rock exhibit.",
  "case_number": "1",
  "exhibition": [
    {
      "uri": "http://radio.liv.ac.uk/bme/Exhibition/Zone6",
      "label": "Exhibition: Madchester & Live Aid (1985-1993)"
    }
  ],
  "image": "https://media.musicarts.com/is/image/MMGS7/L85963000001000-09-2000x2000.jpg",
  "audio": "http://museum.example.org/audio/a4d8c1f

### 2. Search object via string

In [6]:
example_rdf_data = "examples/testset_proto_3.ttl"

def sparql_results_to_json(results):
    """Converts SPARQL query results to a list of dictionaries."""
    return [
        {key: str(value) for key, value in row.items()}
        for row in results.bindings
    ]

In [7]:
from rdflib import Graph
from queries.sparql_queries import SEARCH_OBJECTS_QUERY

g = Graph()
g.parse(example_rdf_data, format="turtle")


<Graph identifier=N235a9fdd1b014d7586514f50a7d7680b (<class 'rdflib.graph.Graph'>)>

In [8]:
def search_objects(query_string: str):
    # Sanitize the input by escaping double quotes and backslashes to prevent SPARQL injection
    query_string = query_string.replace('\\', '\\\\').replace('"', '\\"')
    
    sparql_query = SEARCH_OBJECTS_QUERY % (query_string, query_string, query_string, query_string)
    results = g.query(sparql_query)
    return sparql_results_to_json(results)



In [9]:
search_objects("Liverpool")

[{rdflib.term.Variable('object'): 'http://radio.liv.ac.uk/bme/HumanMadeObject/f8a0d7e4',
  rdflib.term.Variable('description'): 'A pair of round, nickel-rimmed glasses famously worn by John Lennon during the mid-to-late 1960s. This style became an inseparable part of his public image during the psychedelic era and his subsequent solo career.',
  rdflib.term.Variable('name'): "John Lennon's Round 'Granny' Glasses",
  rdflib.term.Variable('image'): 'https://c.files.bbci.co.uk/c24f/live/b4199d60-4e81-11ef-8b97-e7e1e12cdeb2.jpg'}]

### 3. All items in exhibition

In [10]:
from queries.sparql_queries import EXHIBITION_ITEMS_QUERY

ex = "<http://radio.liv.ac.uk/bme/Exhibition/Zone3>"

sparql_query = EXHIBITION_ITEMS_QUERY % ex
results = g.query(sparql_query)

sparql_results_to_json(results)

[{rdflib.term.Variable('description'): "The original 1-inch 4-track analogue master tape for The Beatles' seminal 1967 album, recorded at EMI Studios (now Abbey Road Studios).",
  rdflib.term.Variable('name'): "Master Tape for 'Sgt. Pepper's Lonely Hearts Club Band'",
  rdflib.term.Variable('categoryLabel'): 'Audiovisual Media',
  rdflib.term.Variable('typeLabel'): '1 inch videotape',
  rdflib.term.Variable('image'): 'https://goauctionomega.blob.core.windows.net/stock/25311-2-medium.jpg?v=63798651902413'},
 {rdflib.term.Variable('description'): 'A pair of round, nickel-rimmed glasses famously worn by John Lennon during the mid-to-late 1960s. This style became an inseparable part of his public image during the psychedelic era and his subsequent solo career.',
  rdflib.term.Variable('name'): "John Lennon's Round 'Granny' Glasses",
  rdflib.term.Variable('categoryLabel'): 'Personal Effects',
  rdflib.term.Variable('typeLabel'): 'Spectacles',
  rdflib.term.Variable('image'): 'https://c.fil